# Selective fault tolerance: short GCP pilot
Run one cell at a time: **settings → create → setup → shared profile → pilot → plot → cleanup**.
Start with a fresh kernel and fresh VMs. Use only `main`. The entire experiment has a shared 28-minute work budget, with 90 seconds reserved for stopping the controller. Waiting between cells counts against that budget.

The planner chooses placement, replication and checkpoint frequency using finite enumeration. This is not SMT/Z3, TorchFT, or Fixed-R. Failures kill worker Ray processes and their actors; VMs, the head and shared checkpoint storage survive. Costs/probabilities are modeling assumptions.


In [ ]:
import concurrent.futures
import json
import shlex
import subprocess
import sys
import time
import uuid
from pathlib import Path

PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
NETWORK = "default"  # Existing VPC; no new network is created.
SUBNET = "default"  # Existing subnet in the region of ZONE.
MACHINE_TYPE = "e2-standard-4"
DISK_GB = 100  # Must be at least as large as the saved image.
PYTHON = "/opt/ray-env/ray-dev/bin/python"

TRAIN_DTYPE = "float64"  # Same precision for every policy; reprofile after changing.
FAILURE_MODE = "process"  # Kill Ray + actors; VM stays running. Use "vm" for whole-VM stops.
# One shared 28-minute work budget, including provisioning and profiling.
# Another 90 seconds is reserved for fencing the controller/releasing the lock.
# Starts on the first cloud command; pauses between cells count. Restart kernel for a fresh run.
MAX_EXPERIMENT_MINUTES = 28
FULL_EPOCHS = 600
FULL_CHECKPOINT_EVERY = 100
REPEATS = 1
CASES = [0]  # Predeclared tight case; case 1 is a modeled relaxed-deadline comparison.
PRICE_MULTIPLIERS = [0.25]
FAILURE_PROBABILITIES = [0.5]
DEADLINE_MULTIPLIERS = [1.15, 1.5]
FAULT_TIME_FRACTIONS = [0.1, 0.55]
MAX_OBSERVATIONS = 10  # Two mechanism controls + at most eight comparison runs.
POLICIES = ["checkpoint_retry", "fixed_placement_replication", "placement_first", "joint_enumeration"]
SCENARIOS = ["healthy", "correlated-loss-t0.1"]
INSTALL_MISSING_DEPENDENCIES = True

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gossip_benchmarks").is_dir())
HELPERS = ROOT / "gossip_benchmarks/selective_replication"
WORKDIR = ROOT / "ray_gcp_results"
sys.path.insert(0, str(HELPERS))

# Keep the same names if this cell is rerun in the current kernel.
# Restart the kernel to start a new experiment with new names.
if "RUN_ID" not in globals():
    RUN_ID = "rayexp-" + time.strftime("%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
NAMES = [RUN_ID + "-head"] + [RUN_ID + f"-w{i}" for i in range(1, 5)]
HEAD, *WORKERS = NAMES
REGION = ZONE.rsplit("-", 1)[0]
LOCAL = WORKDIR / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
STATE_FILE = LOCAL / "state.json"
print("VMs:", *NAMES, sep="\n  ")
print("Results:", LOCAL)


## 1. Small command helpers

In [ ]:
def gcloud(*args, timeout=600):
    global EXPERIMENT_DEADLINE
    if "EXPERIMENT_DEADLINE" not in globals():
        EXPERIMENT_DEADLINE = time.monotonic() + MAX_EXPERIMENT_MINUTES * 60
    if not globals().get("CLEANUP_MODE", False):
        remaining = EXPERIMENT_DEADLINE - time.monotonic()
        if remaining <= 0:
            raise TimeoutError("28-minute experiment budget exhausted. Results are preserved; use cleanup below.")
        timeout = min(timeout, remaining)
    command = ["gcloud", *map(str, args), f"--project={PROJECT}", "--quiet"]
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stderr + "\n" + result.stdout)
    return result.stdout


def ssh(name, command):
    return gcloud("compute", "ssh", name, f"--zone={ZONE}", "--tunnel-through-iap",
                  "--ssh-flag=-oConnectTimeout=15", f"--command={command}")


def scp(source, target):
    return gcloud("compute", "scp", source, target, f"--zone={ZONE}", "--tunnel-through-iap")


def run_parallel(function, items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(function, items))


## 2. Create fresh VMs
The CLI check runs before creating billable resources. Rerunning this cell after creation does not create another cluster.
Fresh VMs use the existing `NETWORK` and `SUBNET` selected above. No VPC is created. If `default` is absent, list networks/subnets with `gcloud compute networks list` and `gcloud compute networks subnets list`, then edit the two settings.

In [ ]:
from gcp.capabilities import stop_command

if FAILURE_MODE == "vm":
    print("Failure command:", "gcloud", *stop_command(), "--no-graceful-shutdown")
else:
    print("Failure mode: SIGKILL to worker Ray service and its actors; no VM stop/start.")
if STATE_FILE.exists():
    print("Creation already recorded for this session. Continue with setup.", STATE_FILE)
else:
    image = json.loads(gcloud("compute", "images", "describe", IMAGE, "--format=json"))
    assert image["status"] == "READY"
    gcloud("compute", "networks", "describe", NETWORK, "--format=json")
    subnet = json.loads(gcloud("compute", "networks", "subnets", "describe", SUBNET,
                               f"--region={REGION}", "--format=json"))
    assert subnet["network"].rsplit("/", 1)[-1] == NETWORK, "Subnet belongs to a different VPC"
    # Record names before creation so partial failures can be cleaned up.
    STATE_FILE.write_text(json.dumps(dict(project=PROJECT, zone=ZONE, run_id=RUN_ID,
                                         names=NAMES, image=IMAGE, network=NETWORK, subnet=SUBNET), indent=2))
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-internal",
           f"--network={NETWORK}", "--allow=tcp,udp,icmp",
           f"--source-tags={RUN_ID}", f"--target-tags={RUN_ID}")
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-iap",
           f"--network={NETWORK}", "--allow=tcp:22",
           "--source-ranges=35.235.240.0/20", f"--target-tags={RUN_ID}")
    gcloud("compute", "instances", "create", *NAMES, f"--zone={ZONE}",
           f"--machine-type={MACHINE_TYPE}", f"--image={IMAGE}", f"--image-project={PROJECT}",
           f"--boot-disk-size={max(DISK_GB, int(image['diskSizeGb']))}GB",
           "--boot-disk-type=pd-balanced", "--provisioning-model=STANDARD",
           f"--subnet={SUBNET}", f"--tags={RUN_ID}", f"--labels=ray-experiment={RUN_ID}",
           "--metadata=enable-oslogin=FALSE", "--no-service-account", "--no-scopes", timeout=900)


## 3. Get IPs and wait for SSH

In [ ]:
nodes = []
for name in NAMES:
    vm = json.loads(gcloud("compute", "instances", "describe", name, f"--zone={ZONE}", "--format=json"))
    assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, name
    assert vm["status"] == "RUNNING", (name, vm["status"])
    nodes.append(dict(name=name, ip=vm["networkInterfaces"][0]["networkIP"], head=name == HEAD))
HEAD_IP = nodes[0]["ip"]
print(json.dumps(nodes, indent=2))


def wait_ssh(name):
    for attempt in range(30):
        try:
            ssh(name, "true")
            return
        except (RuntimeError, subprocess.TimeoutExpired):
            if attempt == 29:
                raise
            time.sleep(5)


wait_ssh(HEAD)  # Initialize gcloud's local SSH key before parallel connections.
run_parallel(wait_ssh, WORKERS)


## 4. Install the node script and create the shared directory
The saved image already contains Ray and PyTorch. This cell installs only NFS tools and the small script that starts/stops Ray.

In [ ]:
def setup_node(node):
    name = node["name"]
    ssh(name, """set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-common
id raybench >/dev/null 2>&1 || sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
""")
    path = LOCAL / (name + ".json")
    path.write_text(json.dumps(dict(node, nodes=nodes, head_ip=HEAD_IP, run_id=RUN_ID)))
    scp(path, name + ":/tmp/ray-node.json")
    scp(HELPERS / "gcp/bootstrap/node_control.py", name + ":/tmp/ray-node-control.py")
    ssh(name, """set -e
sudo install -o root -g root -m 644 /tmp/ray-node.json /etc/ray-gcp.json
sudo install -o root -g root -m 644 /tmp/ray-node-control.py /opt/ray-gcp/node_control.py
""")


run_parallel(setup_node, nodes)


## 5. Share checkpoints from the head
Mounts persist when workers stop and restart between observations.

In [ ]:
exports = "/srv/ray-gcp " + " ".join(
    n["ip"] + "(rw,sync,no_subtree_check,root_squash,fsid=0)" for n in nodes[1:]
)
ssh(HEAD, f"""set -e
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nfs-kernel-server
sudo mkdir -p /etc/exports.d
printf '%s\\n' {shlex.quote(exports)} | sudo tee /etc/exports.d/ray-gcp.exports >/dev/null
sudo exportfs -ra
sudo systemctl enable --now nfs-server
""")


def mount_worker(name):
    entry = f"{HEAD_IP}:/ /srv/ray-gcp nfs4 vers=4.2,hard,_netdev 0 0"
    ssh(name, f"""set -e
grep -qF {shlex.quote(entry)} /etc/fstab || printf '%s\\n' {shlex.quote(entry)} | sudo tee -a /etc/fstab >/dev/null
mountpoint -q /srv/ray-gcp || sudo mount /srv/ray-gcp
sudo -u raybench touch /srv/ray-gcp/write-check-{name}
""")


run_parallel(mount_worker, WORKERS)


## 6. Prepare the training runner
`GCPStudy` runs the benchmark and checks recovered model/optimizer/input state. All infrastructure setup above is ordinary notebook code.

In [ ]:
from gcp.launcher import GCPStudy
from gcp.suite import config

cloud = GCPStudy(STATE_FILE, WORKDIR / "selective-vm", failure_mode=FAILURE_MODE)
cloud.experiment_deadline = EXPERIMENT_DEADLINE
cloud.prepare(install_missing_dependencies=INSTALL_MISSING_DEPENDENCIES)


## 7. Profile once and enumerate protection choices
This is a short pilot, not statistical validation. The two shards use 600 and 900 epochs; fine-tuning uses 300. Every policy family gets the same checkpoint options (every 100 epochs or only at task completion), objective, failure model and reliability requirement. Final-only checkpoints mean restart from zero after mid-task loss. Model predictions do not include cloud failure-detection delays.

The objective is expected summed worker busy time subject to the deadline/reliability target. This is a compute-occupancy proxy, not CPU utilization or GCP billing. Fixed placement still optimizes backups and checkpoint intervals; placement-first remains a stronger sequential baseline. We keep ties and infeasible outcomes.


In [ ]:
from gcp.suite import smoke_jobs as mechanism_jobs

FULL = "selective-pilot-01"
full_config = config(epochs=FULL_EPOCHS, checkpoint_every=FULL_CHECKPOINT_EVERY,
                     profile_repeats=1, dtype=TRAIN_DTYPE)
full_config["tasks"]["shard-b"]["epochs"] = 900
full_config["tasks"]["finetune"]["epochs"] = 300
full_config.update(
    keep_all_checkpoints=False,
    objective="expected_worker_busy_s",
    checkpoint_choices={name: sorted({FULL_CHECKPOINT_EVERY, task["epochs"]})
                        for name, task in full_config["tasks"].items()},
    spot_cost_multipliers=PRICE_MULTIPLIERS,
    failure_probabilities=FAILURE_PROBABILITIES,
    deadline_multipliers=DEADLINE_MULTIPLIERS,
    fault_time_fractions=FAULT_TIME_FRACTIONS,
)
searched = cloud.prepare_suite(FULL, full_config)


In [ ]:
# Inspect every predeclared case, including ties and infeasible policy families.
for index, case in enumerate(searched["cases"]):
    print("\nCase", index, case["name"], "deadline", round(case["deadline_s"], 2))
    for policy in POLICIES:
        feasible = case["policies"][policy]
        row = feasible or case["unconstrained_policies"][policy]
        print(policy, "feasible:", feasible is not None,
              "modeled busy seconds:", round(row["expected_worker_busy_s"], 2),
              "modeled elapsed:", round(row["expected_elapsed_s"], 2),
              "deadline success:", row["success_probability"], "plan:", row["plan"])
print("Minutes remaining:", round(cloud.remaining_s() / 60, 1))


## 8. Run the short pilot
Two explicit retry controls first test active single and correlated losses and checkpoint restoration, using the same profile. The comparison then runs four policy families on the same two scenarios (healthy and early correlated loss). Single-worker recovery is tested by the mechanism control; it is not a separate policy comparison in this short pilot. Identical executable plans share observations. One repetition is preliminary evidence. Do not change cases after seeing replay outcomes.

The 28-minute shared budget includes setup, profile and replay; it is a stopping limit, not a promise that all observations will finish. A remote systemd time limit stops the controller even if the notebook disconnects. Completed rows are retained; timed-out comparisons are marked incomplete. VMs remain allocated until the optional cleanup cell is run.


In [ ]:
controls = [j for j in mechanism_jobs(searched)
            if j["policies"] == ["control_retry"]
            and j["scenario"]["name"] in ("control-single", "control-correlated")]
full_jobs = cloud.preview(FULL, repeats=REPEATS, case_indices=CASES, policies=POLICIES)
full_jobs = [j for j in full_jobs if j["scenario"]["name"] in SCENARIOS]
# Controls precede the shuffled comparison; IDs are unique across both groups.
pilot_jobs = [dict(job, id=f"run-{i:04d}") for i, job in enumerate(controls + full_jobs)]
print("Total observations:", len(pilot_jobs), "(including two mechanism controls)")
assert len(controls) == 2 and full_jobs
assert len(pilot_jobs) <= MAX_OBSERVATIONS


In [ ]:
# Stops on a failed mechanism gate, a real training error, or the shared wall budget.
full_summary = cloud.run_suite(FULL, pilot_jobs, mechanism_prefix=2)
print(json.dumps(full_summary, indent=2))


## Plot saved results — run this cell alone
Reads local summary.json and saves completion-time and worker-busy-time plots as PNG/PDF. It works with the current kernel or an explicit RESULTS_DIR; no cluster is needed.


In [ ]:
# Run this cell alone: no VMs, SSH, preparation, or experiment reruns.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = None  # Or Path("/absolute/path/to/full-01") containing summary.json.
PLOT_CASE = 0
if RESULTS_DIR is None:
    if "cloud" in globals() and "FULL" in globals():
        RESULTS_DIR = Path(cloud.local) / FULL
    else:
        root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "gossip_benchmarks").is_dir())
        RESULTS_DIR = root / "ray_gcp_results/selective-vm/selective-pilot-01"
RESULTS_DIR = Path(RESULTS_DIR).expanduser()
summary_path = RESULTS_DIR / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError(f"Set RESULTS_DIR to your saved suite folder: {summary_path}")
report = json.loads(summary_path.read_text())
comparisons = [r for r in report["comparisons"] if r["case"] == PLOT_CASE and not r["policy"].startswith("control_")]
if not comparisons:
    raise ValueError(f"No results for case {PLOT_CASE}")

labels = {
    "round_robin_retry": "Round-robin retry",
    "checkpoint_retry": "Checkpoint retry",
    "fixed_placement_replication": "Fixed replication",
    "placement_first": "Placement first",
    "joint_enumeration": "Selective plan",
}
policies = [p for p in labels if any(r["policy"] == p for r in comparisons)]
policies += sorted({r["policy"] for r in comparisons} - set(policies))
scenarios = sorted({r["scenario"] for r in comparisons},
                   key=lambda s: (s != "healthy", s))
lookup = {(r["scenario"], r["policy"]): r for r in comparisons}
x = np.arange(len(scenarios))
width = 0.8 / len(policies)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), layout="constrained")
for ax, metric, title, ylabel in zip(
    axes,
    ["median_s", "median_worker_busy_s"],
    ["Completion time (lower is better)", "Worker busy time (lower is better)"],
    ["Median elapsed seconds", "Median summed worker busy seconds"],
):
    for i, policy in enumerate(policies):
        values = [lookup.get((s, policy), {}).get(metric, np.nan) for s in scenarios]
        bars = ax.bar(x + (i - (len(policies) - 1) / 2) * width,
                      values, width, label=labels.get(policy, policy))
        ax.bar_label(bars, labels=[f"{v:.1f}" if np.isfinite(v) else "" for v in values],
                     padding=3, fontsize=8)
    ax.set_xticks(x, [s.replace("correlated-loss-", "Correlated loss\n")
                     .replace("loss-spot_a-", "Single worker loss\n")
                     .replace("healthy", "Healthy") for s in scenarios])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(bottom=0)
    ax.margins(y=0.18)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
axes[0].legend(fontsize=8)
counts = sorted({r["n"] for r in comparisons})
fig.suptitle(f"Case {PLOT_CASE} | {', '.join(report.get('failure_modes', []))} loss | "
             f"observations per bar: {counts}")
for suffix in ("png", "pdf"):
    fig.savefig(RESULTS_DIR / f"comparison-case-{PLOT_CASE}.{suffix}", dpi=180)
plt.show()
print("Read:", summary_path)
print("Saved PNG and PDF in:", RESULTS_DIR)
print("Validity:", report["valid"], "| Mechanism coverage:", report.get("mechanism_coverage"))
print("Bars show observed medians, not confidence intervals. One repeat is preliminary evidence.")
print("Injected losses may hit idle workers; active / injected losses below:")
for r in comparisons:
    print(f"  {r['scenario']} | {r['policy']}: "
          f"{r['active_worker_losses']} / {r['injected_worker_losses']}")


## 9. Results
Small JSON results are saved locally after each observation. The optional download also copies remote checkpoint evidence before cleanup.

In [ ]:
print("Results:", cloud.local)
print("Send each suite's profile.json, search.json, replay.json, summary.json, jobs.json and manifest.json")
DOWNLOAD_CHECKPOINTS = False
if DOWNLOAD_CHECKPOINTS:
    for suite in [FULL]:
        print(cloud.download(suite))


## 10. Optional cleanup
Deletes these five VMs and their boot disks, including checkpoints on the head. Download wanted results first. The saved image is retained.

In [ ]:
DELETE_INSTANCES = False
if DELETE_INSTANCES:
    CLEANUP_MODE = True  # Explicit cleanup remains available after the experiment budget.
    try:
        # Exact names and ownership label: never delete unrelated experiment VMs.
        existing = json.loads(gcloud("compute", "instances", "list", "--format=json"))
        targets = [vm for vm in existing if vm["name"] in NAMES and vm["zone"].rsplit("/", 1)[-1] == ZONE]
        for vm in targets:
            assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, vm["name"]
        if targets:
            gcloud("compute", "instances", "delete", *[vm["name"] for vm in targets],
                   f"--zone={ZONE}", "--delete-disks=boot", timeout=900)
        # Remove this run's two firewall rules. Keep the shared network/subnet.
        rules = json.loads(gcloud("compute", "firewall-rules", "list", "--format=json"))
        for rule in rules:
            if rule["name"] in [RUN_ID + "-internal", RUN_ID + "-iap"]:
                assert rule["network"].rsplit("/", 1)[-1] == NETWORK
                gcloud("compute", "firewall-rules", "delete", rule["name"])

    finally:
        CLEANUP_MODE = False
else:
    print("Cleanup skipped. VMs/disks remain billable.")
